In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import col, count, when
from pyspark.sql.functions import year, month, weekofyear
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, round, when





In [0]:
df=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/workspace/default/database/Walmart_Sales.csv")
df.show()

+-----+----------+------------+------------+-----------+----------+-----------+------------+
|Store|      Date|Weekly_Sales|Holiday_Flag|Temperature|Fuel_Price|        CPI|Unemployment|
+-----+----------+------------+------------+-----------+----------+-----------+------------+
|    1|2010-02-05|   1643690.9|           0|      42.31|     2.572|211.0963582|       8.106|
|    1|2010-02-12|  1641957.44|           1|      38.51|     2.548|211.2421698|       8.106|
|    1|2010-02-19|  1611968.17|           0|      39.93|     2.514|211.2891429|       8.106|
|    1|2010-02-26|  1409727.59|           0|      46.63|     2.561|211.3196429|       8.106|
|    1|2010-03-05|  1554806.68|           0|       46.5|     2.625|211.3501429|       8.106|
|    1|2010-03-12|  1439541.59|           0|      57.79|     2.667|211.3806429|       8.106|
|    1|2010-03-19|  1472515.79|           0|      54.58|      2.72| 211.215635|       8.106|
|    1|2010-03-26|  1404429.92|           0|      51.45|     2.732|211

In [0]:
df.count()

6435

In [0]:
df.printSchema()

root
 |-- Store: integer (nullable = true)
 |-- Date: date (nullable = true)
 |-- Weekly_Sales: double (nullable = true)
 |-- Holiday_Flag: integer (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- Fuel_Price: double (nullable = true)
 |-- CPI: double (nullable = true)
 |-- Unemployment: double (nullable = true)



In [0]:
df.select([count(when(col(c).isNull(),c)).alias(c) for c in df.columns]).show()

+-----+----+------------+------------+-----------+----------+---+------------+
|Store|Date|Weekly_Sales|Holiday_Flag|Temperature|Fuel_Price|CPI|Unemployment|
+-----+----+------------+------------+-----------+----------+---+------------+
|    0|   0|           0|           0|          0|         0|  0|           0|
+-----+----+------------+------------+-----------+----------+---+------------+



In [0]:
df_clean=df.dropDuplicates()
df_clean.printSchema()
df_clean.show(5)
print("After clean the row ",df_clean.count())

root
 |-- Store: integer (nullable = true)
 |-- Date: date (nullable = true)
 |-- Weekly_Sales: double (nullable = true)
 |-- Holiday_Flag: integer (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- Fuel_Price: double (nullable = true)
 |-- CPI: double (nullable = true)
 |-- Unemployment: double (nullable = true)

+-----+----------+------------+------------+-----------+----------+-----------+------------+
|Store|      Date|Weekly_Sales|Holiday_Flag|Temperature|Fuel_Price|        CPI|Unemployment|
+-----+----------+------------+------------+-----------+----------+-----------+------------+
|    1|2010-02-12|  1641957.44|           1|      38.51|     2.548|211.2421698|       8.106|
|    1|2010-04-02|  1594968.28|           0|      62.27|     2.719|210.8204499|       7.808|
|    1|2010-05-07|  1603955.12|           0|      72.55|     2.835|210.3399684|       7.808|
|    1|2010-05-14|   1494251.5|           0|      74.78|     2.854|210.3374261|       7.808|
|    1|2010-11-12|

In [0]:
df_features=df_clean.withColumn("year",year(col("Date")))\
                    .withColumn("month",month(col("Date")))\
                    .withColumn("week",weekofyear(col("Date")))
df_features.select("Date","year","month","week").show(10)                                

+----------+----+-----+----+
|      Date|year|month|week|
+----------+----+-----+----+
|2010-02-12|2010|    2|   6|
|2010-04-02|2010|    4|  13|
|2010-05-07|2010|    5|  18|
|2010-05-14|2010|    5|  19|
|2010-11-12|2010|   11|  45|
|2012-05-25|2012|    5|  21|
|2012-08-17|2012|    8|  33|
|2010-02-12|2010|    2|   6|
|2010-10-29|2010|   10|  43|
|2010-11-26|2010|   11|  47|
+----------+----+-----+----+
only showing top 10 rows


In [0]:
store_sales=df_features.groupBy("Store").agg(round(sum("Weekly_sales"),2).alias("Total_sales"))\
                       .orderBy(col("Total_sales").desc())
store_sales.show()


+-----+--------------+
|Store|   Total_sales|
+-----+--------------+
|   20|3.0139779246E8|
|    4|2.9954395338E8|
|   14|2.8899991134E8|
|   13| 2.865177038E8|
|    2|2.7538244098E8|
|   10|2.7161771389E8|
|   27|2.5385591688E8|
|    6|2.2375613064E8|
|    1|2.2240280885E8|
|   39|2.0744554247E8|
|   19| 2.066348621E8|
|   31| 1.996139055E8|
|   23|1.9875061785E8|
|   24|1.9401602128E8|
|   11| 1.939627868E8|
|   28|1.8926368058E8|
|   41|1.8134193489E8|
|   32|1.6681924616E8|
|   18|1.5511473421E8|
|   22|1.4707564857E8|
+-----+--------------+
only showing top 20 rows


In [0]:
monthly_trend = df_features.groupBy("Year", "Month") \
    .agg(round(sum("Weekly_Sales"), 2).alias("Total_Sales")) \
    .orderBy("Year", "Month")

monthly_trend.show(20)

+----+-----+--------------+
|Year|Month|   Total_Sales|
+----+-----+--------------+
|2010|    2|1.9033298304E8|
|2010|    3| 1.819198025E8|
|2010|    4|2.3141236805E8|
|2010|    5|1.8671093434E8|
|2010|    6|1.9224617236E8|
|2010|    7|2.3258012598E8|
|2010|    8|1.8764011089E8|
|2010|    9|1.7726789637E8|
|2010|   10|2.1716182402E8|
|2010|   11|2.0285337014E8|
|2010|   12|2.8876053272E8|
|2011|    1|1.6370396683E8|
|2011|    2|1.8633132787E8|
|2011|    3|1.7935644829E8|
|2011|    4|2.2652651097E8|
|2011|    5|1.8164815816E8|
|2011|    6|1.8977338519E8|
|2011|    7|2.2991139887E8|
|2011|    8|1.8859933225E8|
|2011|    9|2.2084773842E8|
+----+-----+--------------+
only showing top 20 rows


In [0]:
holiday_impact = df_features.groupBy("Holiday_Flag") \
    .agg(round(avg("Weekly_Sales"), 2).alias("Avg_Sales"))

holiday_impact.show()

+------------+----------+
|Holiday_Flag| Avg_Sales|
+------------+----------+
|           1|1122887.89|
|           0|1041256.38|
+------------+----------+



In [0]:
temp_impact=df_features.withColumn("Tempback",floor(col("Temperature")/10)*10)\
                        .groupBy("Tempback")\
                        .agg(round(avg("weekly_sales"),2).alias("Avg_sales")) \
                        .orderBy("Tempback")
temp_impact.show()                        



+--------+----------+
|Tempback| Avg_sales|
+--------+----------+
|     -10| 558027.77|
|       0| 860892.28|
|      10| 848491.15|
|      20|1062941.47|
|      30|1127042.84|
|      40|1113261.78|
|      50|1037328.68|
|      60|1056139.57|
|      70|1062264.37|
|      80| 972035.27|
|      90| 802637.77|
|     100| 289345.67|
+--------+----------+



In [0]:
store_totals = df_features.groupBy("Store") \
    .agg(round(sum("Weekly_Sales"), 2).alias("Total_Sales"))

winow_spec=Window.orderBy(col("Total_sales").desc())
store_rnk=store_totals.withColumn("rank",dense_rank().over(winow_spec))    
store_rnk.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----+--------------+----+
|Store|   Total_Sales|rank|
+-----+--------------+----+
|   20|3.0139779246E8|   1|
|    4|2.9954395338E8|   2|
|   14|2.8899991134E8|   3|
|   13| 2.865177038E8|   4|
|    2|2.7538244098E8|   5|
|   10|2.7161771389E8|   6|
|   27|2.5385591688E8|   7|
|    6|2.2375613064E8|   8|
|    1|2.2240280885E8|   9|
|   39|2.0744554247E8|  10|
|   19| 2.066348621E8|  11|
|   31| 1.996139055E8|  12|
|   23|1.9875061785E8|  13|
|   24|1.9401602128E8|  14|
|   11| 1.939627868E8|  15|
|   28|1.8926368058E8|  16|
|   41|1.8134193489E8|  17|
|   32|1.6681924616E8|  18|
|   18|1.5511473421E8|  19|
|   22|1.4707564857E8|  20|
+-----+--------------+----+
only showing top 20 rows


In [0]:
monthly_store_sales = df_features.groupBy("Store", "Year", "Month") \
    .agg(round(sum("Weekly_Sales"), 2).alias("Monthly_Sales")) \
    .orderBy("Store", "Year", "Month")
mom_window = Window.partitionBy("Store").orderBy("Year", "Month")
monthly_growth = monthly_store_sales.withColumn(
    "Prev_Month_Sales", lag("Monthly_Sales").over(mom_window)
).withColumn(
    "MoM_Growth_Pct",
    round(
        when(col("Prev_Month_Sales").isNotNull(),
             ((col("Monthly_Sales") - col("Prev_Month_Sales")) / col("Prev_Month_Sales")) * 100
        ), 2
    )
)

monthly_growth.show(20)


+-----+----+-----+-------------+----------------+--------------+
|Store|Year|Month|Monthly_Sales|Prev_Month_Sales|MoM_Growth_Pct|
+-----+----+-----+-------------+----------------+--------------+
|    1|2010|    2|    6307344.1|            NULL|          NULL|
|    1|2010|    3|   5871293.98|       6307344.1|         -6.91|
|    1|2010|    4|   7422801.92|      5871293.98|         26.43|
|    1|2010|    5|   5929938.64|      7422801.92|        -20.11|
|    1|2010|    6|   6084081.46|      5929938.64|           2.6|
|    1|2010|    7|   7244483.04|      6084081.46|         19.07|
|    1|2010|    8|   6075952.95|      7244483.04|        -16.13|
|    1|2010|    9|   5829793.92|      6075952.95|         -4.05|
|    1|2010|   10|   7150641.75|      5829793.92|         22.66|
|    1|2010|   11|   6485547.06|      7150641.75|          -9.3|
|    1|2010|   12|   8876953.18|      6485547.06|         36.87|
|    1|2011|    1|   5480050.97|      8876953.18|        -38.27|
|    1|2011|    2|   6399

In [0]:
from pyspark.sql.functions import avg

# 4-week rolling average of Weekly_Sales, per store, ordered by Date
rolling_window = Window.partitionBy("Store").orderBy("Date").rowsBetween(-3, 0)

df_rolling = df_features.withColumn(
    "Rolling_4wk_Avg_Sales",
    round(avg("Weekly_Sales").over(rolling_window), 2)
)

df_rolling.select("Store", "Date", "Weekly_Sales", "Rolling_4wk_Avg_Sales").show(20)

+-----+----------+------------+---------------------+
|Store|      Date|Weekly_Sales|Rolling_4wk_Avg_Sales|
+-----+----------+------------+---------------------+
|    1|2010-02-05|   1643690.9|            1643690.9|
|    1|2010-02-12|  1641957.44|           1642824.17|
|    1|2010-02-19|  1611968.17|           1632538.84|
|    1|2010-02-26|  1409727.59|           1576836.03|
|    1|2010-03-05|  1554806.68|           1554614.97|
|    1|2010-03-12|  1439541.59|           1504011.01|
|    1|2010-03-19|  1472515.79|           1469147.91|
|    1|2010-03-26|  1404429.92|            1467823.5|
|    1|2010-04-02|  1594968.28|            1477863.9|
|    1|2010-04-09|  1545418.53|           1504333.13|
|    1|2010-04-16|  1466058.28|           1502718.75|
|    1|2010-04-23|  1391256.12|            1499425.3|
|    1|2010-04-30|  1425100.71|           1456958.41|
|    1|2010-05-07|  1603955.12|           1471592.56|
|    1|2010-05-14|   1494251.5|           1478640.86|
|    1|2010-05-21|  1399662.

In [0]:
pivot_table=df_features.groupBy("store")\
                       .pivot("year")\
                       .agg(round(sum("weekly_sales"),2))
pivot_table.show(20)                           

+-----+--------------+--------------+-------------+
|store|          2010|          2011|         2012|
+-----+--------------+--------------+-------------+
|    1|   7.3278832E7| 8.092191883E7|6.820205802E7|
|    2| 9.527786419E7| 9.860788142E7|8.149669537E7|
|    3|   1.8745419E7| 2.081687657E7| 1.80244395E7|
|    4| 9.568047081E7|1.1109229333E8|9.277118924E7|
|    5| 1.483603077E7|    1.647082E7|1.416883813E7|
|    6| 7.691232069E7| 8.052876295E7|  6.6315047E7|
|    7| 2.556807815E7| 3.066264052E7|2.536755647E7|
|    8| 4.320447484E7| 4.751278616E7|3.923392013E7|
|    9| 2.512921976E7| 2.868596965E7|2.397402958E7|
|   10| 9.447220221E7| 9.891689474E7|7.822861694E7|
|   11| 6.525513823E7| 7.052358289E7|5.818406568E7|
|   12| 4.837038386E7| 5.258200057E7|4.333484572E7|
|   13| 9.527273545E7|1.0453751333E8|8.670745502E7|
|   14|1.0546224238E8| 1.060962707E8|7.744139826E7|
|   15| 3.202352831E7|  3.22826249E7|2.482753071E7|
|   16| 2.472863259E7| 2.742136749E7|2.210242532E7|
|   17| 4.11

In [0]:
# 2. Top 5 and Bottom 5 performing stores (overall total sales)
store_totals = df_features.groupBy("Store") \
    .agg(round(sum("Weekly_Sales"), 2).alias("Total_Sales"))

print("Top 5 Stores:")
store_totals.orderBy(col("Total_Sales").desc()).show(5)

print("Bottom 5 Stores:")
store_totals.orderBy(col("Total_Sales").asc()).show(5)

Top 5 Stores:
+-----+--------------+
|Store|   Total_Sales|
+-----+--------------+
|   20|3.0139779246E8|
|    4|2.9954395338E8|
|   14|2.8899991134E8|
|   13| 2.865177038E8|
|    2|2.7538244098E8|
+-----+--------------+
only showing top 5 rows
Bottom 5 Stores:
+-----+-------------+
|Store|  Total_Sales|
+-----+-------------+
|   33|3.716022196E7|
|   44|4.329308784E7|
|    5| 4.54756889E7|
|   36|5.341221497E7|
|   38|5.515962642E7|
+-----+-------------+
only showing top 5 rows


In [0]:
# 3. Year-wise growth comparison across stores (using the pivot table)
# Assuming years present are like 2010, 2011, 2012 - adjust based on your actual data
pivot_table.select(
    "Store",
    round(((col("2011") - col("2010")) / col("2010")) * 100, 2).alias("Growth_2010_to_2011"),
    round(((col("2012") - col("2011")) / col("2011")) * 100, 2).alias("Growth_2011_to_2012")
).show(20)

+-----+-------------------+-------------------+
|Store|Growth_2010_to_2011|Growth_2011_to_2012|
+-----+-------------------+-------------------+
|    1|              10.43|             -15.72|
|    2|                3.5|             -17.35|
|    3|              11.05|             -13.41|
|    4|              16.11|             -16.49|
|    5|              11.02|             -13.98|
|    6|                4.7|             -17.65|
|    7|              19.93|             -17.27|
|    8|               9.97|             -17.42|
|    9|              14.15|             -16.43|
|   10|                4.7|             -20.91|
|   11|               8.07|              -17.5|
|   12|               8.71|             -17.59|
|   13|               9.72|             -17.06|
|   14|                0.6|             -27.01|
|   15|               0.81|             -23.09|
|   16|              10.89|              -19.4|
|   17|              12.86|             -13.16|
|   18|              -3.15|             

In [0]:
from pyspark.sql.functions import round, avg, corr

# 1. Average sales by Unemployment level (bucketed)
from pyspark.sql.functions import floor

unemployment_impact = df_features.withColumn(
    "Unemployment_Bucket", floor(col("Unemployment"))
).groupBy("Unemployment_Bucket") \
 .agg(round(avg("Weekly_Sales"), 2).alias("Avg_Sales")) \
 .orderBy("Unemployment_Bucket")

unemployment_impact.show(20)

+-------------------+----------+
|Unemployment_Bucket| Avg_Sales|
+-------------------+----------+
|                  3| 2147430.7|
|                  4|1361415.45|
|                  5|1078752.01|
|                  6| 910691.46|
|                  7|1174871.63|
|                  8|1091821.24|
|                  9| 869670.14|
|                 10| 739738.77|
|                 11| 910441.04|
|                 12| 971108.27|
|                 13| 890313.39|
|                 14| 891879.03|
+-------------------+----------+



In [0]:
# 2. Fuel Price impact on sales trend
fuel_impact = df_features.withColumn(
    "Fuel_Price_Bucket", round(col("Fuel_Price"), 1)
).groupBy("Fuel_Price_Bucket") \
 .agg(round(avg("Weekly_Sales"), 2).alias("Avg_Sales")) \
 .orderBy("Fuel_Price_Bucket")

fuel_impact.show(20)

+-----------------+----------+
|Fuel_Price_Bucket| Avg_Sales|
+-----------------+----------+
|              2.5| 945630.36|
|              2.6| 949612.56|
|              2.7| 986486.39|
|              2.8|1070493.87|
|              2.9| 1135397.6|
|              3.0|1039516.58|
|              3.1|1069997.16|
|              3.2|1131120.75|
|              3.3|1021666.84|
|              3.4|1093906.45|
|              3.5| 1029732.4|
|              3.6|1048057.15|
|              3.7|1044039.95|
|              3.8| 1021067.9|
|              3.9|1052654.97|
|              4.0|1100250.62|
|              4.1|1048537.65|
|              4.2|1067741.31|
|              4.3| 915961.57|
|              4.4| 844021.75|
+-----------------+----------+
only showing top 20 rows
